<a href="https://colab.research.google.com/github/rileykinlaw99-alt/Health-Encryption-Data-Project/blob/main/Encrypted_Health_Status_%26_Sickness_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import base64
import json
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes

In [2]:
class EncryptedHealthTracker:
    def __init__(self, password: str):
        # In production, salt is unique per record
        self.salt = b'secure_random_salt_123'
        self.key = self._derive_key(password)
        self.cipher = AESGCM(self.key)

    def _derive_key(self, password: str) -> bytes:
        """Derives a 256-bit AES key from a user-provided password using PBKDF2."""
        kdf = PBKDF2HMAC(
            algorithm=hashes.SHA256(),
            length=32,
            salt=self.salt,
            iterations=100000,
        )
        return kdf.derive(password.encode('utf-8'))

    def encrypt_record(self, health_data: dict) -> str:
        """Encrypts a health log record and returns a Base64 string (nonce + ciphertext)."""
        # Convert record to JSON bytes
        data_bytes = json.dumps(health_data).encode('utf-8')

        # Generate a unique 12-byte nonce for AES-GCM
        nonce = os.urandom(12)

        # Encrypt data (AES-GCM includes authentication tag automatically)
        ciphertext = self.cipher.encrypt(nonce, data_bytes, None)

        # Combine nonce and ciphertext, then encode as Base64 string
        payload = nonce + ciphertext
        return base64.b64encode(payload).decode('utf-8')

    def decrypt_record(self, encrypted_record: str) -> dict:
        """Decrypts and authenticates an encrypted health log record."""
        # Decode Base64 payload
        payload = base64.b64decode(encrypted_record.encode('utf-8'))

        # Extract 12-byte nonce and ciphertext
        nonce = payload[:12]
        ciphertext = payload[12:]

        # Decrypt and authenticate
        decrypted_bytes = self.cipher.decrypt(nonce, ciphertext, None)
        return json.loads(decrypted_bytes.decode('utf-8'))


if __name__ == "__main__":
    password = "SuperSecretPassword123!"
    tracker = EncryptedHealthTracker(password)

    sample_record = {
        "patient_id": "P-90210",
        "status": "Recovering",
        "symptoms": ["Fever", "Cough"],
        "temp_celsius": 37.8
    }

    # Encrypt
    encrypted_payload = tracker.encrypt_record(sample_record)
    print("Encrypted Payload (Base64):", encrypted_payload)

    # Decrypt
    decrypted_record = tracker.decrypt_record(encrypted_payload)
    print("Decrypted Record:", decrypted_record)



Encrypted Payload (Base64): Gq3eyG9+xdmg7/plvo4hcYgZvQcxb5p2lKpN2hLEhUfNLru2xsGWD8y8OzD+LdpZlhddp+kaY+B2vqBwXe8d93ks432xcfQ7rdEnficES6eBil/2xPCiFnE0AUmN7EXGMq28ZLvV3QsO5zoIXTmhna3Wxy2qDznjw5MvqX99VetCmm8=
Decrypted Record: {'patient_id': 'P-90210', 'status': 'Recovering', 'symptoms': ['Fever', 'Cough'], 'temp_celsius': 37.8}
